In [1]:
import sqlite3
import pandas as pd
import pantab
from pathlib import Path

folder = Path(r"C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined db")


In [2]:
db_paths = list(folder.glob("*.db"))
#print (db_paths)

def load_texts_from_dbs(db_paths):
    dfs = {}

    for db_path in db_paths:
        with sqlite3.connect(db_path) as conn:
            dfs[db_path.stem] = pd.read_sql_query(
                'SELECT u, id, ts, text, TURKU_NLP, TURKU_NLP_sub, Tags, "web-register" FROM texts;',
                conn
            )

    return dfs

dfs = load_texts_from_dbs(db_paths)


In [3]:
print(dfs["Combined_single"].columns)

Index(['u', 'id', 'ts', 'text', 'Turku_NLP', 'Turku_NLP_sub', 'Tags',
       'web-register'],
      dtype='object')


In [4]:
dfs_noNA = {
    name: df.dropna(subset=["Turku_NLP", "Turku_NLP_sub"])
    for name, df in dfs.items()
}

for name, df in dfs_noNA.items():
    print(name, df.shape)

Combined_hybrid (6194, 8)
Combined_ID_hybrid (6194, 8)
Combined_single (6194, 8)
Combined_SP_hybrid (6194, 8)


# Remove NSFW/HATE from data

In [5]:
search_term = "NSFW"

results = []

for dataset_name, df in dfs_noNA.items():
    matches = df[df["Tags"].astype(str).str.contains(search_term, case=False, na=False)].copy()

    if not matches.empty:
        matches["Dataset"] = dataset_name
        results.append(matches)

# Combine all matching rows into one DataFrame
if results:
    results_df = pd.concat(results, ignore_index=True)
   #print(results_df)
else:
    print("No matches found.")

In [6]:
search_terms = ["HATE", "NSFW"]

pattern = "|".join(search_terms) 

dfs_filtered = {}

for dataset_name, df in dfs_noNA.items():
    dfs_filtered[dataset_name] = df[
        ~df["Tags"].astype(str).str.contains(pattern, case=False, na=False)
    ].copy()

In [7]:
for dataset_name, df in dfs_filtered.items():
    print(f"{dataset_name}: {df.shape}")

Combined_hybrid: (6185, 8)
Combined_ID_hybrid: (6185, 8)
Combined_single: (6185, 8)
Combined_SP_hybrid: (6185, 8)


In [8]:
#Sanity check
search_terms = ["HATE", "NSFW"]
pattern = "|".join(search_terms)

for dataset_name, df in dfs_filtered.items():
    n_matches = df["Tags"].astype(str).str.contains(
        pattern, case=False, na=False
    ).sum()

    print(f"{dataset_name}: {n_matches} matches remaining")

Combined_hybrid: 0 matches remaining
Combined_ID_hybrid: 0 matches remaining
Combined_single: 0 matches remaining
Combined_SP_hybrid: 0 matches remaining


In [9]:
dfs_filtered["Combined_single"].head()

,u,id,ts,text,Turku_NLP,Turku_NLP_sub,Tags,web-register
0,http://roshangari.info/?p=35434,110ae9e775485f536c4f1cdea087cbd7,2020-01-25T19:50:48Z,e.shafagh@yahoo.com\n“تاریخ گواهی خواهد داد که...,IP ; ID,ed ; -,OTHLI,"{""MT"": 0.074, ""LY"": 0.084, ""SP"": 0.105, ""ID"": ..."
1,https://www.tinn.ir/%D8%A8%D8%AE%D8%B4-%D9%88%...,32962503dcdb08ee1bc4d2e9d5240bc7,2020-01-19T18:40:03Z,در کشورهای پیشرفته که حملونقل بر اساس پژوهشهای...,IP,ed,None,"{""MT"": 0.089, ""LY"": 0.085, ""SP"": 0.125, ""ID"": ..."
2,http://p313.ir/post-124289.html,0e6562306fcb8db55e019445407cb487,2016-10-26T00:51:16Z,مهدی محمدی طی یادداشتی در روزنامه وطن امروز نو...,IP,ed,None,"{""MT"": 0.093, ""LY"": 0.063, ""SP"": 0.11900000000..."
3,http://shakhesnews.com/%db%b6-%d8%af%d8%a7%d9%...,588d3acafdc57f5122849f3163c1d357,2016-10-27T18:45:54Z,شاخص : اگر مسیر مذاکرات در همین جهت ادامه یابد...,IP,ed,OTHLI,"{""MT"": 0.092, ""LY"": 0.07100000000000001, ""SP"":..."
4,https://rabnews.ir/8350/%D8%AF%D8%B1-%D8%AC%D8...,9afaecf7d64a78e22fa71dec39be048b,2020-01-24T00:25:08Z,نظام آموزشی ایران؛ علم زنده یا مرده؟\nدر جامعه...,IP ; SP,ed ; os,OTHLI,"{""MT"": 0.082, ""LY"": 0.09, ""SP"": 0.269, ""ID"": 0..."


# Registers

In [10]:
def process_turku(df, main_col="Turku_NLP", sub_col="Turku_NLP_sub"):
    """
    Processes a dataframe by splitting the given Turku columns,
    pairing elementwise, exploding, and fixing '-' entries.
    """

    df_copy = df.copy()

    # Step 1: Split into lists
    df_copy[f"{main_col}_split"] = df_copy[main_col].str.split(" ; ")
    df_copy[f"{sub_col}_split"] = df_copy[sub_col].str.split(" ; ")

    # Step 2: Pair elementwise
    df_copy["paired"] = df_copy.apply(
        lambda x: list(zip(x[f"{main_col}_split"], x[f"{sub_col}_split"])),
        axis=1
    )

    # Step 3: Explode
    exploded = df_copy.explode("paired").copy()

    # Step 4: Restore into two separate columns
    exploded[f"{main_col}_split"] = exploded["paired"].apply(
        lambda x: x[0] if pd.notna(x) else None
    )
    exploded[f"{sub_col}_split"] = exploded["paired"].apply(
        lambda x: x[1] if pd.notna(x) else None
    )

    exploded = exploded.drop(columns=["paired"])

    # Step 5: Replace "-" in sub column
    exploded[f"{sub_col}_split_modified"] = exploded[f"{sub_col}_split"]

    mask = (
        (exploded[f"{sub_col}_split"] == "-")
        & (exploded[f"{main_col}_split"] != "-")
    )

    exploded.loc[mask, f"{sub_col}_split_modified"] = exploded.loc[
        mask, f"{main_col}_split"
    ]

    return exploded

In [11]:
dfs_processed = {}

for name, df in dfs_filtered.items():
    if df.empty:
        continue

    processed = process_turku(df)
    processed["source"] = name
    dfs_processed[name] = processed

#df_all = pd.concat(dfs_processed.values(), ignore_index=True)

In [12]:
#df_all = pd.concat(dfs_processed.values(), ignore_index=True)

In [13]:
print(dfs_processed["Combined_single"]["Turku_NLP_sub_split_modified"].unique())
print(dfs_processed["Combined_single"]["Turku_NLP_split"].unique())

['ed' 'ID' 'os' 'on' 'ob' 'ne' 'ra' 'rs' 'dtp' 'oo' 'LY' 'MT' 'oe' 'oi'
 'lt' 'en' 're' 'fi' 'av' 'it' 'oh' 'ds' 'nb' 'rv' '-' 'sr']
['IP' 'ID' 'SP' 'NA' 'OP' 'IN' 'LY' 'MT' 'HI' '-']


In [14]:
print(len((dfs_processed["Combined_single"]["Turku_NLP_sub_split_modified"].unique()))) #should be 26
print(len(dfs_processed["Combined_single"]["Turku_NLP_split"].unique())) #should be 10

26
10


In [15]:
# Find erros like lt ;oh (where space after ; forgotten)
def find_value(dfs_processed, column, value):
    for name, df in dfs_processed.items():
        matches = df[df[column] == value]

        if not matches.empty:
            print(f"\n===== {name} ({len(matches)} matches) =====")
            display(matches)

In [16]:
find_value(
    dfs_processed,
    "Turku_NLP_sub_split_modified", #Turku_NLP_split
    "ob , on" #for example lt ;oh
)

In [17]:
dfs_processed["Combined_single"].tail()
#Turku_NLP_split is the main register
#Turku_NLP_sub_split is the sub register
#Turku_NLP_sub_split_modified for subregister "-" --> replaced by main register

,u,id,ts,text,Turku_NLP,Turku_NLP_sub,Tags,web-register,Turku_NLP_split,Turku_NLP_sub_split,Turku_NLP_sub_split_modified,source
6191,http://kivili.mihanblog.com/post/278,e73b912d54da19268fe75066c03e720e,2018-05-20T13:28:50Z,جام ورزشی، شهبازی در خصوص وضعیت انصاری فرد و ح...,NA,sr,None,"{""MT"": 0.088, ""LY"": 0.063, ""SP"": 0.146, ""ID"": ...",NA,sr,sr,Combined_single
6192,https://www.esteghlalnews.com/%D8%A7%D8%AE%D8%...,b9a0f6ac982f809c8f7f0e8f086cfab4,2021-04-18T09:10:48Z,سرمربی تیم ملی گفت: من در بازی فینال لیگ قهرما...,NA ; SP,sr ; os,None,"{""MT"": 0.098, ""LY"": 0.094, ""SP"": 0.34700000000...",NA,sr,sr,Combined_single
6192,https://www.esteghlalnews.com/%D8%A7%D8%AE%D8%...,b9a0f6ac982f809c8f7f0e8f086cfab4,2021-04-18T09:10:48Z,سرمربی تیم ملی گفت: من در بازی فینال لیگ قهرما...,NA ; SP,sr ; os,None,"{""MT"": 0.098, ""LY"": 0.094, ""SP"": 0.34700000000...",SP,os,os,Combined_single
6193,https://arteshesorkh.com/post/%D9%85%D9%86%D8%...,ec77d02c7e15c673d5b5db803880fc37,2024-06-18T14:42:10Z,نویسنده : ارتش سرخ دات کام\n-\nنظرات : 9 اظهار...,SP ; ID,it ; -,None,"{""MT"": 0.115, ""LY"": 0.116, ""SP"": 0.52, ""ID"": 0...",SP,it,it,Combined_single
6193,https://arteshesorkh.com/post/%D9%85%D9%86%D8%...,ec77d02c7e15c673d5b5db803880fc37,2024-06-18T14:42:10Z,نویسنده : ارتش سرخ دات کام\n-\nنظرات : 9 اظهار...,SP ; ID,it ; -,None,"{""MT"": 0.115, ""LY"": 0.116, ""SP"": 0.52, ""ID"": 0...",ID,-,ID,Combined_single


In [18]:
dfs_processed["Combined_single"].nunique()

u                               6057
id                              6160
ts                              6157
text                            6160
Turku_NLP                        194
Turku_NLP_sub                    502
Tags                              11
web-register                    6111
Turku_NLP_split                   10
Turku_NLP_sub_split               23
Turku_NLP_sub_split_modified      26
source                             1
dtype: int64

## Check Inconsistent IDs

In [19]:
# Columns you want to compare
cols_to_check = ["Turku_NLP", "Turku_NLP_sub"]

# Group by ID
grouped = dfs_processed["Combined_hybrid"].groupby("id") #Combined_ID_hybrid #Combined_single #Combined_SP_hybrid #same for last line

inconsistent_ids = []

for doc_id, group in grouped:
    # For each of the columns, check if there is more than one unique non-null value
    inconsistent = False
    for col in cols_to_check:
        unique_vals = group[col].dropna().unique()
        if len(unique_vals) > 1:
            inconsistent = True
    if inconsistent:
        inconsistent_ids.append(doc_id)

print("Inconsistent IDs:", inconsistent_ids)
print(f"Total inconsistent IDs: {len(inconsistent_ids)}")

# Optional: print detailed info like Claude did
for doc_id in inconsistent_ids:
    print("\nID", doc_id, "has inconsistent values:")
    display(dfs_processed["Combined_ID_hybrid"][dfs_processed["Combined_ID_hybrid"]["id"] == doc_id][["id", "Turku_NLP", "Turku_NLP_sub", "source"]])


Inconsistent IDs: []
Total inconsistent IDs: 0


In [20]:
doc_id = "3625e84eed5ba426fc397d6c489143ae"

group = dfs_processed["Combined_SP_hybrid"][
    dfs_processed["Combined_SP_hybrid"]["id"] == doc_id
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    for val in group[col].dropna().unique():
        print("value:", repr(val))
        print("type:", type(val))
        print("length:", len(str(val)))
        print("characters:", [f"{c} (U+{ord(c):04X})" for c in str(val)])


--- Turku_NLP ---
value: 'SP ; IP'
type: <class 'str'>
length: 7
characters: ['S (U+0053)', 'P (U+0050)', '  (U+0020)', '; (U+003B)', '  (U+0020)', 'I (U+0049)', 'P (U+0050)']

--- Turku_NLP_sub ---
value: 'os ; ed'
type: <class 'str'>
length: 7
characters: ['o (U+006F)', 's (U+0073)', '  (U+0020)', '; (U+003B)', '  (U+0020)', 'e (U+0065)', 'd (U+0064)']


## CSV

In [21]:
import os

output_dir = r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output"
os.makedirs(output_dir, exist_ok=True)

for name, df in dfs_processed.items():
    file_path = os.path.join(output_dir, f"{name}.csv")
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

Saved: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\Combined_hybrid.csv
Saved: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\Combined_ID_hybrid.csv
Saved: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\Combined_single.csv
Saved: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\Combined_SP_hybrid.csv


In [22]:
dfs_processed["Combined_single"].tail()

,u,id,ts,text,Turku_NLP,Turku_NLP_sub,Tags,web-register,Turku_NLP_split,Turku_NLP_sub_split,Turku_NLP_sub_split_modified,source
6191,http://kivili.mihanblog.com/post/278,e73b912d54da19268fe75066c03e720e,2018-05-20T13:28:50Z,جام ورزشی، شهبازی در خصوص وضعیت انصاری فرد و ح...,NA,sr,None,"{""MT"": 0.088, ""LY"": 0.063, ""SP"": 0.146, ""ID"": ...",NA,sr,sr,Combined_single
6192,https://www.esteghlalnews.com/%D8%A7%D8%AE%D8%...,b9a0f6ac982f809c8f7f0e8f086cfab4,2021-04-18T09:10:48Z,سرمربی تیم ملی گفت: من در بازی فینال لیگ قهرما...,NA ; SP,sr ; os,None,"{""MT"": 0.098, ""LY"": 0.094, ""SP"": 0.34700000000...",NA,sr,sr,Combined_single
6192,https://www.esteghlalnews.com/%D8%A7%D8%AE%D8%...,b9a0f6ac982f809c8f7f0e8f086cfab4,2021-04-18T09:10:48Z,سرمربی تیم ملی گفت: من در بازی فینال لیگ قهرما...,NA ; SP,sr ; os,None,"{""MT"": 0.098, ""LY"": 0.094, ""SP"": 0.34700000000...",SP,os,os,Combined_single
6193,https://arteshesorkh.com/post/%D9%85%D9%86%D8%...,ec77d02c7e15c673d5b5db803880fc37,2024-06-18T14:42:10Z,نویسنده : ارتش سرخ دات کام\n-\nنظرات : 9 اظهار...,SP ; ID,it ; -,None,"{""MT"": 0.115, ""LY"": 0.116, ""SP"": 0.52, ""ID"": 0...",SP,it,it,Combined_single
6193,https://arteshesorkh.com/post/%D9%85%D9%86%D8%...,ec77d02c7e15c673d5b5db803880fc37,2024-06-18T14:42:10Z,نویسنده : ارتش سرخ دات کام\n-\nنظرات : 9 اظهار...,SP ; ID,it ; -,None,"{""MT"": 0.115, ""LY"": 0.116, ""SP"": 0.52, ""ID"": 0...",ID,-,ID,Combined_single


# Plots

In [23]:
import pandas as pd
import plotly.express as px
import os


labels_structure = {
    "MT": ["-"],
    "LY": ["-"],
    "SP": ["it" , "os"],
    "ID": ["-"],
    "NA": ["ne", "sr", "nb", "on"],
    "HI": ["re" , "oh"],
    "IN": ["en", "ra", "dtp", "fi", "lt", "oi"],
    "OP": ["rv", "ob", "rs", "av", "oo"],
    "IP": ["ds", "ed", "oe"],
    "-" : ["-"] #no register
}



# ---------------------------------------------------------
# 2. Process one dataframe
# ---------------------------------------------------------

def process_hierarchy(df, parent_col="Turku_NLP", child_col="Turku_NLP_sub"):
    
    records = []

    for _, row in df.iterrows():

        parents = (
            str(row[parent_col]).split(";")
            if pd.notna(row[parent_col])
            else []
        )

        children = (
            str(row[child_col]).split(";")
            if pd.notna(row[child_col])
            else []
        )

        # Clean whitespace
        parents = [x.strip() for x in parents if x.strip()]
        children = [x.strip() for x in children if x.strip()]

        # -------------------------------------------------
        # Pair parent and child according to their position
        # -------------------------------------------------

        if len(parents) != len(children):
            records.append({
                "id": row["id"],
                "parent": None,
                "child": None,
                "valid": False,
                "problem": (
                    f"Number of parents ({len(parents)}) != "
                    f"number of children ({len(children)})"
                )
            })
            continue

        for parent, child in zip(parents, children):

            # Is this child allowed under this parent?
            valid = (
                parent in labels_structure
                    and child in labels_structure[parent]
                )

            if parent not in labels_structure:
                problem = f"Unknown parent label: {parent}"

            elif child not in labels_structure[parent]:
                # Find where this child IS allowed, if anywhere
                possible_parents = [
                    p for p, children_list in labels_structure.items()
                    if child in children_list
            ]

                if possible_parents:
                    problem = (
                        f"{child} is defined under {possible_parents}, "
                        f"but was assigned to {parent}"
                )
                else:
                    problem = f"Unknown sublabel: {child}"

            else:
                problem = None

            records.append({
                "id": row["id"],
                "parent": parent,
                "child": child,
                "valid": valid,
                "problem": problem
        })

    return pd.DataFrame(records)

In [24]:
processed_hierarchy = {}

for dataset_name, df in dfs_processed.items():

    result = process_hierarchy(df)

    processed_hierarchy[dataset_name] = result

In [25]:
for dataset_name, df in processed_hierarchy.items():

    errors = df[~df["valid"]]

    if len(errors) > 0:

        print("\n" + "=" * 80)
        print(f"ERRORS IN: {dataset_name}")
        print("=" * 80)

        display(
            errors[
                ["id", "parent", "child", "problem"]
            ]
        )

In [26]:
def plot_hierarchy(df, dataset_name):

    # Only valid combinations for the normal chart
    valid_df = df[df["valid"]].copy()

    if valid_df.empty:
        print(f"No valid data to plot for {dataset_name}")
        return

    counts = (
        valid_df
        .groupby(["parent", "child"])["id"]
        .nunique()
        .reset_index(name="count")
    )

    fig = px.bar(
        counts,
        x="parent",
        y="count",
        color="child",
        title=f"{dataset_name} — Hierarchical Label Distribution",
        text_auto=True,
        category_orders={
            "parent": list(labels_structure.keys())
        }
    )

    fig.update_layout(
        barmode="stack",
        xaxis_title="Parent label",
        yaxis_title="Count",
        legend_title="Sub-label",
        template="plotly_white"
    )

    fig.show()

In [27]:
import plotly.io as pio

pio.renderers.default = "browser"


In [28]:
for dataset_name, df in processed_hierarchy.items():
    plot_hierarchy(df, dataset_name)

# With no NA data

In [33]:
import os
import pandas as pd

# Path where you want the CSV files saved
output_path = r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\without_NA"

# Create the output directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)



In [34]:
# Store the cleaned copies here, if you also want them in memory
dfs_processed_cleaned = {}

for dataset_name, df in dfs_processed.items():
    
    # Make a copy so the original DataFrame is not modified
    df_clean = df.copy()

    # Remove rows where BOTH columns contain "-"
    mask = (
        df_clean["Turku_NLP"].eq("-") &
        df_clean["Turku_NLP_sub"].eq("-")
    )
    
    df_clean = df_clean.loc[~mask].copy()

    # Store the cleaned DataFrame
    dfs_processed_cleaned[dataset_name] = df_clean

    # Save as CSV
    output_file = os.path.join(output_path, f"{dataset_name}_no_NA.csv")
    df_clean.to_csv(output_file, index=False, encoding="utf-8")

    print(
        f"{dataset_name}: "
        f"{len(df)} rows -> {len(df_clean)} rows | "
        f"removed {mask.sum()} rows"
    )

Combined_hybrid: 9198 rows -> 7243 rows | removed 1955 rows
Combined_ID_hybrid: 8651 rows -> 6696 rows | removed 1955 rows
Combined_single: 8255 rows -> 6300 rows | removed 1955 rows
Combined_SP_hybrid: 8809 rows -> 6854 rows | removed 1955 rows


# Split the data

# Finding some issues

In [29]:
def get_sp_ids(dataset_name):
    df = processed_hierarchy[dataset_name]

    return {
        "it": set(
            df.loc[
                (df["parent"] == "SP") &
                (df["child"] == "it") &
                (df["valid"]),
                "id"
            ]
        ),
        "os": set(
            df.loc[
                (df["parent"] == "SP") &
                (df["child"] == "os") &
                (df["valid"]),
                "id"
            ]
        )
    }


for dataset_name in processed_hierarchy:
    sp = get_sp_ids(dataset_name)

    print(
        dataset_name,
        "SP → it:", len(sp["it"]),
        "SP → os:", len(sp["os"])
    )

Combined_hybrid SP → it: 214 SP → os: 194
Combined_ID_hybrid SP → it: 214 SP → os: 194
Combined_single SP → it: 214 SP → os: 194
Combined_SP_hybrid SP → it: 214 SP → os: 194


In [30]:
id_sp = get_sp_ids("Combined_ID_hybrid")
sp_sp = get_sp_ids("Combined_SP_hybrid")

In [31]:
print("SP → it: in others but NOT Combined_ID_hybrid")
print(sp_sp["it"] - id_sp["it"])

print("\nSP → os: in others but NOT Combined_ID_hybrid")
print(sp_sp["os"] - id_sp["os"])

SP → it: in others but NOT Combined_ID_hybrid
set()

SP → os: in others but NOT Combined_ID_hybrid
set()


In [32]:
print("\nSP → it: in Combined_ID_hybrid but NOT Combined_SP_hybrid")
print(id_sp["it"] - sp_sp["it"])

print("\nSP → os: in Combined_ID_hybrid but NOT Combined_SP_hybrid")
print(id_sp["os"] - sp_sp["os"])


SP → it: in Combined_ID_hybrid but NOT Combined_SP_hybrid
set()

SP → os: in Combined_ID_hybrid but NOT Combined_SP_hybrid
set()


# DO not Run Below
## Tableau

In [83]:
df_Tableau_single = dfs_processed["Combined_single"].copy()
df_Tableau_single.loc[:, ['id', 'Turku_NLP_split', 'Turku_NLP_sub_split_modified', 'source', 'Tags']]

,id,Turku_NLP_split,Turku_NLP_sub_split_modified,source,Tags
0,110ae9e775485f536c4f1cdea087cbd7,IP,ed,Combined_single,OTHLI
0,110ae9e775485f536c4f1cdea087cbd7,ID,ID,Combined_single,OTHLI
1,32962503dcdb08ee1bc4d2e9d5240bc7,IP,ed,Combined_single,None
2,0e6562306fcb8db55e019445407cb487,IP,ed,Combined_single,None
3,588d3acafdc57f5122849f3163c1d357,IP,ed,Combined_single,OTHLI
...,...,...,...,...,...
6191,e73b912d54da19268fe75066c03e720e,NA,sr,Combined_single,None
6192,b9a0f6ac982f809c8f7f0e8f086cfab4,NA,sr,Combined_single,None
6192,b9a0f6ac982f809c8f7f0e8f086cfab4,SP,os,Combined_single,None
6193,ec77d02c7e15c673d5b5db803880fc37,SP,it,Combined_single,None


In [ ]:
df_Tableau_single.to_excel(
    "Single_data_for_tableau.xlsx",
    index=False,
    engine="openpyxl"
)


In [36]:
# find errors from Tableu chart

def find_rows(df, split ="OP", sub_split = "dtp"):
    result = df[
        (df["Turku_NLP_split"] == split) &
        (df["Turku_NLP_sub_split_modified"] == sub_split)
    ]
    
    if result.empty:
        print("No matching rows found.")
        return None

    return result

In [37]:
#find_rows(df_Tableau, "OP", "ed")